In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rfa_model as rfa

In [2]:
%matplotlib widget

## Load STLs

In [3]:
stl_dir = Path("../rfa stl")

meshes, sample_parts = rfa.load_and_align_sample_assembly(
    stl_dir,
    alpha_deg=0.0,
)

frame_meshes, frame_parts, frame_info = rfa.load_and_align_grid_frames(
    stl_dir,
    verbose=True,
)

sample_y_bounds, sample_z_bounds = rfa.sample_bounds(meshes)

Grid frame alignment
  fitted center: [-0.00113118 -0.00140144 -0.00109564]
  fitted radius: 0.05887084242766213
  axis before alignment: [ 9.99990913e-01  4.23261006e-03 -5.08542373e-04]
  R_g1: 0.04519042252564098
  R_g2: 0.05792646490184952
  R_g3: 0.07107616160072151


## Load field

In [4]:
field = rfa.load_field_npz("../../rfa_field_sample_0_g2g3_0_collector50_0p5mm.npz")

print(field.keys())
print("V shape:", field["V"].shape)
print("h:", field["h"])

dict_keys(['x', 'y', 'z', 'h', 'V', 'Ex', 'Ey', 'Ez', 'R_g1', 'R_g2', 'R_g3', 'fixed', 'update_region', 'Vfix', 'owner', 'voltages', 'Vs', 'Vr', 'Vg1', 'Vg2', 'Vg3', 'Vc', 'Vdt'])
V shape: (333, 333, 333)
h: 0.0005


In [5]:
voltages = {
    "Vs": 0.0,
    "Vr": 0.0,
    "Vg1": 0.0,
    "Vg2": 0.0,
    "Vg3": 0.0,
    "Vc": 50.0,
    "Vdt": 0.0,
}

In [6]:
field["voltages"] = voltages
field["R_g1"] = frame_info["R_g1"]
field["R_g2"] = frame_info["R_g2"]
field["R_g3"] = frame_info["R_g3"]
field["R_col"] = 0.08255

In [7]:
Ex_interp, Ey_interp, Ez_interp = rfa.build_field_interpolators(field)
Phi_interp = rfa.build_potential_interpolator(field)

p_test = np.array([0.001, 0.0, 0.0])

In [8]:
field = rfa.attach_default_owner_name_map(field)

## Build intersectors

In [9]:
collision_meshes_primary = rfa.build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=True,
)

collision_mesh_primary, face_owner_primary, intersector_primary = rfa.build_stl_intersector(
    collision_meshes_primary
)

stl_boxes_primary = rfa.build_stl_bounding_boxes(
    collision_meshes_primary,
    padding=1.0e-3,
)

collision_meshes_emit = rfa.build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=False,
)

collision_mesh_emit, face_owner_emit, intersector_emit = rfa.build_stl_intersector(
    collision_meshes_emit
)

stl_boxes_emit = rfa.build_stl_bounding_boxes(
    collision_meshes_emit,
    padding=1.0e-3,
)

## Samplers

In [10]:
model_dir = Path("../data/jmonsel")
bronstein_dir = Path("../data/bronstein")

yield_models, energy_models, theta_models = rfa.load_default_surface_models(
    model_dir=model_dir,
    bronstein_dir=bronstein_dir,
)

## Run trajectories

In [ ]:
grid_transparency = {
    "g1_shell": 0.93,
    "g2_shell": 0.93,
    "g3_shell": 0.93,
}


cascade = rfa.run_cascade_batch_parallel(
    N_primary=100,
    E0_eV=500,
    field=field,
    Phi_interp=Phi_interp,
    Ex_interp=Ex_interp,
    Ey_interp=Ey_interp,
    Ez_interp=Ez_interp,

    intersector_primary=intersector_primary,
    face_owner_primary=face_owner_primary,
    collision_mesh_primary=collision_mesh_primary,
    stl_boxes_primary=stl_boxes_primary,

    intersector_emit=intersector_emit,
    face_owner_emit=face_owner_emit,
    collision_mesh_emit=collision_mesh_emit,
    stl_boxes_emit=stl_boxes_emit,

    grid_transparency=grid_transparency,

    yield_models=yield_models,
    energy_models=energy_models,
    theta_models=theta_models,
    voltages=voltages,
    SEY_mult=1.0,

    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,

    max_generation=6,
    max_total_electrons_per_primary=300,
    min_incident_energy_eV=0.5,

    emitted_max_step_fraction_of_h=0.75,
    emitted_dt_max=5.0e-11,
    emitted_max_steps=40000,
    launch_step_fraction_of_h=0.75,

    seed=123,
    n_jobs=4,
    chunk_size=5,
    verbose=10,
)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   40.7s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  1.4min


In [ ]:
cascade["df_cascade"]["reason"].value_counts()
cascade["current_counts"]["net_count"].sum()

## Data analysis

### Load CSV results

In [ ]:
result_dir = Path("../results")
prefix = "cascade_500eV_h025mm_N100000_T09_gen6"  # adjust name

df_cascade = pd.read_csv(result_dir / f"{prefix}_cascade.csv")
df_primary = pd.read_csv(result_dir / f"{prefix}_primary.csv")
current_counts = pd.read_csv(result_dir / f"{prefix}_current_counts.csv", index_col=0)
summary = pd.read_csv(result_dir / f"{prefix}_summary.csv").iloc[0].to_dict()

N_primary = int(summary["N_primary"])

print("N primary:", N_primary)
print("N cascade electrons:", len(df_cascade))
current_counts

In [ ]:
df_cascade = cascade["df_cascade"]
df_primary = cascade["df_primary"]
current_counts = cascade["current_counts"]
summary = cascade["summary"]
N_primary = cascade["N_primary"]

In [14]:
ELECTRODE_ORDER = [
    "sample",
    "holder",
    "receiver",
    "rod",
    "grid1",
    "grid2",
    "grid3",
    "collector",
    "drifttube",
    "escaped",
    "unknown",
]


def primary_arrival_per_primary(df_primary, N_primary):
    """
    Primary arrival contribution to sample current.

    For normal runs this should be 1.0, because every primary hits sample.
    """
    if df_primary is None or len(df_primary) == 0:
        return 1.0

    if "reason" in df_primary.columns:
        return float((df_primary["reason"] == "hit_sample").sum()) / float(N_primary)

    return 1.0


def matlab_like_net_per_primary(current_counts, df_primary=None, N_primary=None):
    """
    Convert Python cascade current_counts to MATLAB-style arrived-emitted counts.

    Python:
        net_per_primary = source - terminal

    MATLAB:
        net_count = arrived - emitted

    For emitted/cascade electrons:
        MATLAB-like = -Python

    Then add primary arrival to sample.
    """
    if "net_per_primary" not in current_counts.columns:
        raise ValueError("current_counts must contain net_per_primary")

    if N_primary is None:
        raise ValueError("N_primary is required")

    out = -current_counts["net_per_primary"].copy()
    out = out.reindex(ELECTRODE_ORDER, fill_value=0.0)

    out["sample"] += primary_arrival_per_primary(df_primary, N_primary)

    return out


def true_yields_from_cascade(df_cascade, N_primary):
    """
    True primary-induced sample yields from cascade table.

    Generation 1 = emissions caused by the primary sample hit.
    """
    gen1_sample = df_cascade[
        (df_cascade["generation"] == 1)
        & (df_cascade["source_electrode"] == "sample")
    ]

    true_tey_primary = len(gen1_sample)

    true_bse_primary = int(
        gen1_sample["emission_kind"].astype(str).isin(
            ["BSE", "quantum_reflection", "QR"]
        ).sum()
    )

    true_tey = true_tey_primary / N_primary
    true_bse = true_bse_primary / N_primary

    return {
        "true_tey_primary": true_tey_primary,
        "true_bse_primary": true_bse_primary,
        "true_tey": true_tey,
        "true_bse": true_bse,
    }


def predict_mode_from_cascade(
    df_cascade,
    df_primary,
    current_counts,
    N_primary,
    mode="TEY",
):
    """
    MATLAB-like measured-current prediction and correction factor.

    Valid modes:
        TEY
        sample_bias_BSE
        grid_bias_BSE
    """
    mode = str(mode)

    if mode not in ["TEY", "sample_bias_BSE", "grid_bias_BSE"]:
        raise ValueError("mode must be TEY, sample_bias_BSE, or grid_bias_BSE")

    matlab_net = matlab_like_net_per_primary(
        current_counts,
        df_primary=df_primary,
        N_primary=N_primary,
    )

    measured_sample_side = (
        matlab_net["sample"]
        + matlab_net["holder"]
        + matlab_net["receiver"]
    )

    measured_rfa_all = (
        matlab_net["collector"]
        + matlab_net["drifttube"]
        + matlab_net["rod"]
        + matlab_net["grid1"]
        + matlab_net["grid2"]
        + matlab_net["grid3"]
    )

    measured_rfa_after_retarding = (
        matlab_net["collector"]
        + matlab_net["grid2"]
        + matlab_net["grid3"]
    )

    measured_total_current = measured_sample_side + measured_rfa_all

    y = true_yields_from_cascade(df_cascade, N_primary)

    if mode == "TEY":
        true_yield = y["true_tey"]
        true_primary_count = y["true_tey_primary"]
        measured_yield = measured_rfa_all / measured_total_current

    elif mode == "sample_bias_BSE":
        true_yield = y["true_bse"]
        true_primary_count = y["true_bse_primary"]
        measured_yield = measured_rfa_all / measured_total_current

    elif mode == "grid_bias_BSE":
        true_yield = y["true_bse"]
        true_primary_count = y["true_bse_primary"]
        measured_yield = measured_rfa_after_retarding / measured_total_current

    correction_factor = true_yield / measured_yield

    return {
        "mode": mode,
        "N_primary": N_primary,

        "true_tey_primary": y["true_tey_primary"],
        "true_bse_primary": y["true_bse_primary"],
        "true_primary_count_for_mode": true_primary_count,

        "measured_sample_side": measured_sample_side * N_primary,
        "measured_total_current": measured_total_current * N_primary,
        "measured_rfa_all": measured_rfa_all * N_primary,
        "measured_rfa_after_retarding": measured_rfa_after_retarding * N_primary,

        "measured_sample_side_per_primary": measured_sample_side,
        "measured_total_current_per_primary": measured_total_current,
        "measured_rfa_all_per_primary": measured_rfa_all,
        "measured_rfa_after_retarding_per_primary": measured_rfa_after_retarding,

        "true_yield": true_yield,
        "measured_yield": measured_yield,
        "correction_factor": correction_factor,
    }


def predict_all_modes_from_cascade(
    df_cascade,
    df_primary,
    current_counts,
    N_primary,
):
    rows = []

    for mode in ["TEY", "sample_bias_BSE", "grid_bias_BSE"]:
        rows.append(
            predict_mode_from_cascade(
                df_cascade=df_cascade,
                df_primary=df_primary,
                current_counts=current_counts,
                N_primary=N_primary,
                mode=mode,
            )
        )

    return pd.DataFrame(rows)

### Prediction table

In [ ]:
df_pred = predict_all_modes_from_cascade(
    df_cascade=df_cascade,
    df_primary=df_primary,
    current_counts=current_counts,
    N_primary=N_primary,
)

cols = [
    "mode",
    "N_primary",
    "true_tey_primary",
    "true_bse_primary",
    "measured_sample_side",
    "measured_total_current",
    "measured_rfa_all",
    "measured_rfa_after_retarding",
    "true_yield",
    "measured_yield",
    "correction_factor",
]

df_pred[cols]

### Current distribution table

In [ ]:
matlab_net = matlab_like_net_per_primary(
    current_counts,
    df_primary=df_primary,
    N_primary=N_primary,
)

df_current = pd.DataFrame({
    "python_source_minus_terminal_per_primary": current_counts["net_per_primary"],
    "matlab_arrived_minus_emitted_per_primary": matlab_net,
})

df_current["matlab_arrived_minus_emitted_count"] = (
    df_current["matlab_arrived_minus_emitted_per_primary"] * N_primary
)

df_current

### Plot predicted MATLAB-like current distribution

In [ ]:
ax = df_current["matlab_arrived_minus_emitted_per_primary"].plot(
    kind="bar",
    figsize=(8, 4),
)

ax.axhline(0, linewidth=0.8)
ax.set_ylabel("Arrived - emitted electrons per primary")
ax.set_title("Predicted MATLAB-like electrode current balance")
plt.tight_layout()

### Print summary: sample bias

In [ ]:
pred_sample_bias = predict_mode_from_cascade(
    df_cascade=df_cascade,
    df_primary=df_primary,
    current_counts=current_counts,
    N_primary=N_primary,
    mode="sample_bias_BSE",
)

for k, v in pred_sample_bias.items():
    print(f"{k:35s}: {v}")

### Print summary: grid bias

In [ ]:
pred_grid_bias = predict_mode_from_cascade(
    df_cascade=df_cascade,
    df_primary=df_primary,
    current_counts=current_counts,
    N_primary=N_primary,
    mode="grid_bias_BSE",
)

for k, v in pred_grid_bias.items():
    print(f"{k:35s}: {v}")